# 1006 24일차

## 1. 문서 분할

긴 문서를 검색 단위인 청크로 자르는 것

문서 전체를 벡터 하나로 만들면 질문과 관련 있는 부분이 어디인지 구분할 수 없음. 청크마다 벡터를 만들어야 필요한 부분만 찾아옴

1. `TextLoader` : 텍스트 파일 하나를 `Document` 하나로 불러옴. `metadata`의 `source`에 파일 경로, `page_content`에 본문
2. `RecursiveCharacterTextSplitter` : 구분자 순서대로 문서를 청크로 자름
3. `load_and_split(splitter)` : 불러오기와 분할을 한 번에

- 청크도 `Document`라 원본의 `source`가 그대로 붙음
- samsung·nvidia 문서가 각각 청크 9개, 합쳐서 18개

### 1-1. 분할 규칙

| 인자 | 값 | 뜻 |
|---|---|---|
| `chunk_size` | 300 | 청크 길이 상한 (글자 수) |
| `chunk_overlap` | 100 | 앞 청크 끝부분을 다음 청크 앞에 겹쳐 넣는 길이 |
| `separators` | `["\n\n", "\n", " ", ""]` | 자르는 기준. 기본값과 같음 |

1. `separators[0]`부터 순서대로 자르고, `chunk_size`를 넘는 조각만 다음 구분자로 다시 자름
2. 짧은 조각은 `chunk_size` 안에서 이웃끼리 합침
3. `""`는 글자 단위. 공백 없이 긴 덩어리만 여기까지 가고, 이때만 `chunk_size`에 딱 맞게 잘림

- 길이를 300에 맞추기보다 문단·문장을 깨지 않는 쪽을 우선함. 그래서 청크 길이가 제각각
- samsung 문서 : 문단 8개(제목 포함) → 청크 9개 `[185, 219, 239, 235, 298, 299, 170, 187, 249]`
  - 제목(10자)은 첫 문단과 합쳐짐 (185)
  - 219자·239자 문단은 합치면 300을 넘어 따로 남음
  - 571자 스마트폰 문단만 단어 단위로 다시 잘려 298·299·170
- 겹침은 경계에서 잘린 문장을 다음 청크에도 남겨 검색할 때 문맥을 놓치지 않게 함
- 다음 청크로 넘어가는 건 앞 청크 끝의 `chunk_overlap`(100자) 이하 조각뿐. samsung 문서는 스마트폰 문단 안에서만 겹침이 생김

### 1-2. 여러 파일 불러오기

`glob`으로 파일 목록을 만들고 파일마다 `TextLoader`를 돌려 `Document`를 한 리스트에 모음

1. `glob(os.path.join(path, "*.txt"))` : 폴더 안 txt 파일 경로 리스트
2. `data += loader.load()` : `load()`가 돌려준 `[Document]`의 원소를 이어 붙임
3. `split_documents(data)` : 모은 `Document` 리스트를 한 번에 분할

- `append`를 쓰면 `[[Doc], [Doc], [Doc]]`처럼 리스트 안에 리스트가 들어가 splitter에서 에러
- 파일 3개(8158·2049·1898자) → 청크 52개 (`chunk_overlap=10`). 청크마다 원본 파일 경로가 `source`로 붙음
- 청크 길이를 볼 때 `(len(...) for ...)`는 제너레이터라 print하면 `<generator object>`만 나옴. `list()`로 꺼내야 값이 보임

#### cf) 합치기와 겹침

| | 합치기 | 겹침 |
|---|---|---|
| 하는 일 | 짧은 조각 여러 개를 청크 하나로 묶음 | 앞 청크 끝부분을 다음 청크 앞에 한 번 더 넣음 |
| 같은 내용이 들어가는 청크 | 1개 | 2개 |
| samsung 문서 | 제목 + 첫 문단 → 청크 1개 | 5번 청크 끝 98자가 6번 청크 앞에도 |

```
합치기 : 청크1 = A B C  /  청크2 = D E
겹침   : 청크1 = A B C  /  청크2 = C D E
```

## 2. 청크 임베딩

청크 하나가 통째로 벡터 하나가 됨. 토큰마다 벡터가 되는 keras `Embedding`과 다름 (day22 RAG §1-1)

1. `embed_query(문자열)` : 문자열 하나 → 벡터 하나. 검색할 때 질문을 임베딩 (day22 RAG §1)
2. `embed_documents(리스트)` : 문자열 리스트 → 벡터 리스트. `from_documents`가 청크를 저장할 때 씀

- 청크 N개 → `(N, dimensions)`. 청크 18개, `dimensions=5`면 `(18, 5)`
- 청크 길이와 상관없이 벡터 길이는 `dimensions`로 같음. `dimensions`를 빼면 1536
- 모델 안에서는 토큰 단위로 처리하지만 나오는 건 청크당 벡터 하나

## 3. Chroma

청크·벡터·메타데이터를 폴더에 저장해 두고 다시 불러 쓰는 벡터 DB (day23 RAG §1-1)

한 번 저장해 두면 검색할 때마다 문서를 다시 임베딩하지 않아도 됨

| | 저장 | 불러오기 |
|---|---|---|
| 코드 | `Chroma.from_documents()` | `Chroma()` |
| 하는 일 | 청크를 임베딩해서 컬렉션에 추가 | 저장된 컬렉션에 연결 |
| 임베딩 인자 | `embedding` | `embedding_function` |

1. `persist_directory` : DB를 저장하는 폴더
2. `collection_name` : 저장할 때와 같은 이름이어야 함. 틀리면 에러 없이 빈 컬렉션이 새로 생김

- 불러올 때 넘기는 임베딩 모델은 질문을 벡터로 바꾸는 데 씀. 저장할 때와 같은 모델·`dimensions`여야 비교 가능
- 불러오기는 연결만 함. 데이터는 `get()`, `similarity_search()`를 부를 때 읽음

### 3-1. 저장된 데이터 확인

`db.get()` : 저장된 청크를 dict로 돌려줌

| 키 | 내용 |
|---|---|
| `ids` | 청크마다 붙는 id |
| `documents` | 청크 원문 |
| `metadatas` | 청크마다 `source` |
| `embeddings` | `None` |

- 기본으로 `metadatas`·`documents`만 가져옴. 벡터까지 보려면 `get(include=['embeddings'])`
- `db._collection.count()` : 컬렉션에 들어 있는 전체 청크 수. 실행할 때마다 누적

#### cf) 다시 실행하면 쌓이는 문서

`from_documents`는 실행할 때마다 같은 청크를 새 id로 추가함

- 저장을 두 번 실행해 18개 → 36개. 앞 18개와 뒤 18개는 내용이 같고 id만 다름
- 검색 결과에도 같은 청크가 두 번씩 나옴
- 새로 저장하려면 DB 폴더를 지우고 한 번만 실행

## 4. 유사도 검색

`similarity_search(질문, k)` : 질문과 가장 가까운 청크 `k`개를 `Document` 리스트로 돌려줌. 기본 `k=4`

질문도 청크를 저장할 때와 같은 모델로 임베딩해서 청크 벡터와 거리를 잼 (의미 검색 흐름은 day23 RAG §1)

- 결과는 가까운 순이고 `Document`에 `id`가 같이 나옴
- "삼성전자 사업전망에 대해 알려줘" → samsung 청크 4개. 중복 저장 때문에 서로 다른 청크는 2개 (§3 cf)

## 5. Retriever

질문을 넣으면 문서를 돌려주도록 벡터 DB를 감싼 검색기

다른 LangChain 구성 요소처럼 `invoke`로 불러서 LCEL 체인에 그대로 연결할 수 있음 (day21 RAG §3)

1. `as_retriever(search_kwargs={"k": 2})` : 벡터 DB를 retriever로 바꿈. `search_kwargs`에 검색 설정, `k`는 돌려줄 문서 수
2. `retriever.invoke(질문)` : 가까운 청크 `k`개를 `Document` 리스트로

| | `similarity_search` | retriever |
|---|---|---|
| 호출 | `db.similarity_search(질문, k=4)` | `retriever.invoke(질문)` |
| `k` 지정 | 부를 때마다 | 만들 때 한 번 |

- "삼성전자의 창업자는 누구인가요?" → 청크 2개, 첫 번째는 "삼성전자 사업 전망" 청크
- 문서에 창업자 내용이 없어도 가장 가까운 청크 `k`개를 돌려줌. 답이 들어 있는지는 보장하지 않음